# ImageICL, from scratch

The image side of the model has one stage that involves labels: **ImageICL**, the image in-context read (the class is `LabelledRead` in `table_image_icl/model.py`). Every patch of every photo attends to the patches of the labelled context photos. The queries and keys are patch tokens, so content decides where to look. The values are **label embeddings**, so what comes back is a mix of the labels of the photos it looked at. What a patch reads is added to its token, and the tokens then join their row in TabICL's row stage; the prediction itself is made later, by TabICL's in-context stage.

This notebook builds that up from nothing, one step per cell:

- **Steps 1–5:** a dot product, a softmax, labels as the values, and why that is in-context learning.
- **Steps 6–13:** adding the vote into the photo's own token, one small step at a time.
- **Step 14:** how those tokens enter TabICL's row stage.

An appendix shows how the query and key projections learn *what* to compare across many small tasks.

Each photo starts as one small vector. Nothing here needs real images.

In [1]:
import os, sys
os.chdir(next(p for p in [os.getcwd(), os.path.dirname(os.getcwd())] if os.path.exists(os.path.join(p, "table_image_icl"))))
sys.path.insert(0, os.getcwd())
import torch
import torch.nn.functional as F
torch.manual_seed(0); torch.set_printoptions(precision=2, sci_mode=False, linewidth=120)

## 1. A tiny table of photos

Six context photos of animals with labels, and one test photo without. Each photo is a 4-number vector, standing in for what the encoder makes of it. To keep it readable, each number is how strongly the photo shows one thing:

| | stripes | spots | wings | four legs |
|---|---|---|---|---|
| zebra (label 0) | 2 or 1 | 0 | 0 | 2 |
| bird (label 1) | 0 | 0 or 1 | 2 | 0 |
| leopard (label 2) | 0 | 2 or 1 | 0 | 2 or 1 |

The test photo is a bird whose legs show a little: wings 2, four legs 1. Its label should be 1, but the model never sees it.

In [2]:
#                        stripes spots wings legs
context = torch.tensor([[2.,     0.,   0.,   2.],      # zebra    label 0
                        [1.,     0.,   0.,   2.],      # zebra    label 0
                        [0.,     0.,   2.,   0.],      # bird     label 1
                        [0.,     1.,   2.,   0.],      # bird     label 1
                        [0.,     2.,   0.,   2.],      # leopard  label 2
                        [0.,     1.,   0.,   1.]])     # leopard  label 2
labels = torch.tensor([0, 0, 1, 1, 2, 2])
test = torch.tensor([0., 0., 2., 1.])                  # a bird, legs showing

print("context photos", tuple(context.shape), "(photos, width)")
print(context)
print("labels        ", labels.tolist())
print("test photo    ", test)

context photos (6, 4) (photos, width)
tensor([[2., 0., 0., 2.],
        [1., 0., 0., 2.],
        [0., 0., 2., 0.],
        [0., 1., 2., 0.],
        [0., 2., 0., 2.],
        [0., 1., 0., 1.]])
labels         [0, 0, 1, 1, 2, 2]
test photo     tensor([0., 0., 2., 1.])


## 2. Scores: how much does the test photo look like each context photo?

The simplest similarity is a dot product. One score per context photo:

$$s_j = x_{\text{test}} \cdot x_j$$

The two birds share the wings and score 4. Because the test bird's legs show, every four-legged animal also gets a little score (1 or 2): the dot product counts every feature the two photos share.

In [3]:
scores = context @ test                     # (6,)
for j, (s, y) in enumerate(zip(scores, labels)):
    print(f"photo {j}  label {y.item()}  score {s.item():4.1f}")

photo 0  label 0  score  2.0
photo 1  label 0  score  2.0
photo 2  label 1  score  4.0
photo 3  label 1  score  4.0
photo 4  label 2  score  2.0
photo 5  label 2  score  1.0


## 3. Weights: a softmax over the scores

Attention turns scores into weights that are positive and add up to 1. Before the softmax the scores are divided by $\sqrt{d}$, with $d$ the width (4 here, so by 2), as in every transformer; it keeps the softmax from saturating when $d$ is large. The birds get about 0.30 each; the rest share what is left.

$$w_j = \frac{\exp(s_j / \sqrt{d})}{\sum_{j'} \exp(s_{j'} / \sqrt{d})}$$

In [4]:
d = context.shape[1]
weights = torch.softmax(scores / d ** 0.5, dim=0)
print("weights", weights)
print("sum    ", weights.sum().item())

weights tensor([0.11, 0.11, 0.30, 0.30, 0.11, 0.07])
sum     1.0


## 4. The values: ordinary attention, then labels

In ordinary self-attention the output is a weighted average of the *same* vectors you compared with. Here that gives back something that looks mostly like a bird (lots of wings), but nothing about the label itself:

$$o = \sum_j w_j \, x_j$$

In [5]:
ordinary = weights @ context
print("weighted average of the photos:", ordinary)
print("   (stripes, spots, wings, legs)")

weighted average of the photos: tensor([0.33, 0.59, 1.20, 0.73])
   (stripes, spots, wings, legs)


ImageICL swaps the values. Keys stay the photos, so the weights are the same, but each context photo passes on **its label** instead of its content. With one-hot labels, $e_y$ has a 1 at position $y$:

$$r = \sum_j w_j \, e_{y_j}$$

Every context photo votes for its own label, as strongly as the test photo attends to it. The result is a distribution over labels.

In [6]:
n_labels = 3
E = F.one_hot(labels, n_labels).float()       # (6, 3): one row per context photo
print("label embeddings E (values)")
print(E)
read = weights @ E
print("read r =", read, " -> predicted label", read.argmax().item())

label embeddings E (values)
tensor([[1., 0., 0.],
        [1., 0., 0.],
        [0., 1., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [0., 0., 1.]])
read r = tensor([0.22, 0.60, 0.18])  -> predicted label 1


`weights @ E` is the same as adding up the weights per label, which you can check by hand:

In [7]:
by_hand = torch.zeros(n_labels)
for w, y in zip(weights, labels):
    by_hand[y] += w
print(by_hand)

tensor([0.22, 0.60, 0.18])


## 5. Why this is in-context learning

Nothing in steps 2–4 depends on which label means what. Relabel the context, so the birds are now label 2 and the leopards label 1, and the same computation predicts 2 for the same test photo. No weight was trained or changed; the answer comes from the labels that came in with the context.

This is also why the model's labels are *slots* rather than class names: a task's classes land on random slots, so the model cannot memorise that slot 1 means "cat", and has to read the answer off the context.

In [8]:
new_labels = torch.tensor([0, 0, 2, 2, 1, 1])
new_read = weights @ F.one_hot(new_labels, n_labels).float()
print("old labels", labels.tolist(), "-> read", read, "-> label", read.argmax().item())
print("new labels", new_labels.tolist(), "-> read", new_read, "-> label", new_read.argmax().item())

old labels [0, 0, 1, 1, 2, 2] -> read tensor([0.22, 0.60, 0.18]) -> label 1
new labels [0, 0, 2, 2, 1, 1] -> read tensor([0.22, 0.18, 0.60]) -> label 2


## 6. A token has a fixed width

Step 4 gave the test photo a vote, $(0.22, 0.60, 0.18)$. The rest of the model only takes tokens, so the vote has to go *into* the photo's token. The token does not get longer for it: in the model a patch token is 128 numbers when it enters the read and 128 when it leaves.

So give the toy a fixed width too: 7 numbers per photo. The first 4 are the appearance numbers from step 1. The other 3 we **set to zero ourselves**, to keep space free where the vote can be read at a glance; nothing in steps 1–5 changes, since zeros add nothing to a dot product.

That is a choice for this toy, not something the model does. The real encoder uses all 128 numbers, and the vote is added on top of them. Step 12 shows that case.

In [9]:
WIDTH = 7                                   # stripes spots wings legs, then 3 zeros we keep free for the vote
def token(appearance):
    t = torch.zeros(WIDTH); t[:4] = appearance
    return t

test_token = token(test)
print("test token:", test_token)

test token: tensor([0., 0., 2., 1., 0., 0., 0.])


## 7. Each label becomes a vector of the same width

Each label's embedding is a vector as wide as a token, 7 numbers. Here, zebra is a 1 in place 5, bird in place 6, leopard in place 7: the three places we kept free. These are the values now, instead of the one-hot rows of step 4.

In [10]:
label_vectors = torch.zeros(n_labels, WIDTH)
label_vectors[0, 4] = 1                     # zebra
label_vectors[1, 5] = 1                     # bird
label_vectors[2, 6] = 1                     # leopard
print(label_vectors)

tensor([[0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 1.]])


## 8. The read is the same vote, now token-shaped

Same weights as step 3, same weighted sum, new values. The read is 7 numbers wide, like a token, and the vote sits in the last three:

$$r = \sum_j w_j \, e_{y_j}$$

In [11]:
r = weights @ label_vectors[labels]
print("r =", r)

r = tensor([0.00, 0.00, 0.00, 0.00, 0.22, 0.60, 0.18])


## 9. Add the read to the token

The token keeps what it had and gains what it read: $h = x + r$, 7 numbers plus 7 numbers, still 7. This is the residual connection of any transformer layer, and the same way transformers add position embeddings to word embeddings, or TabICL adds its label embedding to a context row: by adding, not by appending.

In [12]:
h_test = test_token + r
print("x     ", test_token, " width", len(test_token))
print("r     ", r, " width", len(r))
print("x + r ", h_test, " width", len(h_test))

x      tensor([0., 0., 2., 1., 0., 0., 0.])  width 7
r      tensor([0.00, 0.00, 0.00, 0.00, 0.22, 0.60, 0.18])  width 7
x + r  tensor([0.00, 0.00, 2.00, 1.00, 0.22, 0.60, 0.18])  width 7


## 10. The context photos do the same

Every photo reads, the context photos too. They read from the context photos, themselves included, so each mostly reads back its own label. That is allowed: the context labels are inputs. The test photo is never read from, so its label is never used.

In [13]:
tokens = torch.stack([token(x) for x in context] + [test_token])      # 6 context photos, then the test photo
w = torch.softmax(tokens @ tokens[:6].T / 4 ** 0.5, dim=-1)           # every photo against the 6 context photos
h = tokens + w @ label_vectors[labels]

print("          stripes spots wings legs  zebra bird leopard")
for name, t in zip(["zebra", "zebra", "bird", "bird", "leopard", "leopard", "TEST"], h):
    print(f"{name:8s}", t)

          stripes spots wings legs  zebra bird leopard
zebra    tensor([2.00, 0.00, 0.00, 2.00, 0.86, 0.02, 0.12])
zebra    tensor([1.00, 0.00, 0.00, 2.00, 0.73, 0.05, 0.23])
bird     tensor([0.00, 0.00, 2.00, 0.00, 0.11, 0.79, 0.11])
bird     tensor([0.00, 1.00, 2.00, 0.00, 0.08, 0.75, 0.17])
leopard  tensor([0.00, 2.00, 0.00, 2.00, 0.18, 0.05, 0.77])
leopard  tensor([0.00, 1.00, 0.00, 1.00, 0.30, 0.15, 0.56])
TEST     tensor([0.00, 0.00, 2.00, 1.00, 0.22, 0.60, 0.18])


## 11. Change the labels, and only the vote changes

As in step 5, make the birds label 2 and the leopards label 1. The test photo looks the same, so its first four numbers stay put; what the context says about it has moved.

In [14]:
h_new = tokens + w @ label_vectors[new_labels]
print("old labels:", h[6])
print("new labels:", h_new[6])

old labels: tensor([0.00, 0.00, 2.00, 1.00, 0.22, 0.60, 0.18])
new labels: tensor([0.00, 0.00, 2.00, 1.00, 0.22, 0.18, 0.60])


## 12. In the model, nothing is set aside

The tidy layout above, appearance in the first 4 numbers and the vote in the last 3, was our choice, for reading. In the model nothing is kept free: the image encoder uses all 128 numbers, and the label embeddings are 10 learned directions in that same space, not places of their own.

To see that the read works just as well that way, turn the whole toy by a random rotation of the 7-dimensional space: every photo token and every label vector. Now no single number means "wings" or "bird"; every number is a mix.

In [15]:
torch.manual_seed(1)
turn, _ = torch.linalg.qr(torch.randn(WIDTH, WIDTH))                 # a random rotation: turn @ turn.T = I
tokens_mixed, labels_mixed = tokens @ turn, label_vectors @ turn

w_mixed = torch.softmax(tokens_mixed @ tokens_mixed[:6].T / 4 ** 0.5, dim=-1)
h_mixed = tokens_mixed + w_mixed @ labels_mixed[labels]
print("test token, mixed:", h_mixed[6], " width", len(h_mixed[6]))
print("same weights as before:", torch.allclose(w_mixed, w, atol=1e-6))

test token, mixed: tensor([ 0.18,  0.21, -0.06,  0.47, -1.75, -1.38, -0.39])  width 7
same weights as before: True


The numbers look like noise, but nothing is lost. A rotation keeps every dot product, so the weights are the same as in step 10. And the vote can be read back by a dot product with each label's direction, which is what a later layer can learn to do:

$$h \cdot e_c \;=\; \underbrace{x \cdot e_c}_{0} \;+\; r \cdot e_c \;=\; p_c$$

The first term is 0 only because the toy made the appearance exactly perpendicular to the label directions. In the model nothing guarantees that: a photo's content has some part along every label direction (and the directions the vote really lands on are $e_c W_V W_O$, after the value and output projections). That part shows up as crosstalk when a later layer reads the vote back. Training can push the content away from those directions, since crosstalk that looks like a vote would hurt the prediction, but nothing forces it to zero.

In [16]:
print("vote read back, h . e_c:", h_mixed[6] @ labels_mixed.T)
print("vote from step 4:      ", read)

vote read back, h . e_c: tensor([0.22, 0.60, 0.18])
vote from step 4:       tensor([0.22, 0.60, 0.18])


## 13. Patches: every patch reads for itself

In the model a photo is not one token but 256 patch tokens, and each patch reads on its own: its keys are the patches of the context photos, and a key patch's value is the label of the photo it belongs to. So two patches of one photo can read different things.

A bird in front of a zebra, in two patches (the context photos stay one patch each to keep it small):

In [17]:
patches = torch.stack([token(torch.tensor([0., 0., 2., 0.])),       # the bird part
                       token(torch.tensor([2., 0., 0., 1.]))])      # the zebra behind it
w_patches = torch.softmax(patches @ tokens[:6].T / 4 ** 0.5, dim=-1)
patch_tokens = patches + w_patches @ label_vectors[labels]
print("          stripes spots wings legs  zebra bird leopard")
print("bird part ", patch_tokens[0])
print("zebra part", patch_tokens[1])

          stripes spots wings legs  zebra bird leopard
bird part  tensor([0.00, 0.00, 2.00, 0.00, 0.11, 0.79, 0.11])
zebra part tensor([2.00, 0.00, 0.00, 1.00, 0.81, 0.06, 0.13])


Both tokens move on as they are; nothing averages them into one vote.

## 14. Into TabICL's row stage

TabICL turns each table cell into a token, and keeps 4 places at the front of the row for its CLS tokens. Our patch tokens are put **after the cell tokens, in the same row**:

```
row = [CLS × 4 | one token per table column | 256 patch tokens]
```

Before that, a learned **patch marker** is added to every patch token so the row stage can tell patches from cells. Here are the shapes at the model's real sizes, with the real stage: 40 photos (36 context, 4 test), width 128, and a stand-in for TabICL's tokens of a 3-column table.

In [18]:
from table_image_icl.model import LabelledRead
torch.manual_seed(0)
stage = LabelledRead(width=128, heads=4, layers=2, n_slots=10, max_sources=256, read="top_k", top_k=32).eval()
patch_marker = torch.randn(128) * 0.02

image_tokens = torch.randn(1, 40, 256, 128)          # from the image encoder: (batch, photos, patches, width)
y_context = torch.randint(10, (1, 36))               # the 36 context photos' label slots
with torch.no_grad():
    patch_tokens = stage(image_tokens, y_context) + patch_marker
cell_tokens = torch.zeros(1, 40, 4 + 3, 128)         # TabICL's: (batch, rows, CLS + columns, width)
rows = torch.cat([cell_tokens, patch_tokens], dim=2)

print("patch tokens in    ", tuple(image_tokens.shape))
print("patch tokens out   ", tuple(patch_tokens.shape), "  same shape, label vote added")
print("into the row stage ", tuple(rows.shape), "  4 CLS + 3 cells + 256 patches per row")

patch tokens in     (1, 40, 256, 128)
patch tokens out    (1, 40, 256, 128)   same shape, label vote added
into the row stage  (1, 40, 263, 128)   4 CLS + 3 cells + 256 patches per row


From here it is TabICL's own machinery: 3 transformer blocks over each row (cells and patches attend to each other), the 4 CLS outputs joined into the row's vector (4 × 128 = 512), and the in-context stage, which adds TabICL's own label embedding to the context rows and predicts the test rows. Nothing in the read has a loss of its own; it is all trained end to end through that prediction.

## What the real stage adds to the toy

The toy is the core. The real stage, `LabelledRead`, adds, in order:

- **Learned label vectors.** 10 label slots, each a learned direction in the token's 128 dimensions, as in step 12; they start orthogonal to each other.
- **Projections.** Queries and keys go through $W_Q$ and $W_K$ (learned: *what* to compare, see the appendix); the label vectors through $W_V$ and $W_O$. Those are linear, so the read is still the vote, over 10 learned label directions.
- **4 heads**, each with its own weights and its own vote.
- **LayerNorms** before each read.
- **An MLP, then a second read.** The MLP mixes appearance and vote in one token. In the second read the queries carry the vote; the keys are still the label-free tokens.
- **The 32 nearest context photos.** Each photo reads only the patches of the 32 context photos most like it (mean patch token, cosine).

## Check: the real code, set to the toy

Set `LabelledRead` to one head, one read, the identity for every projection, no LayerNorm, the MLP at zero, and our label vectors. It gives back the tokens of step 10. (PyTorch divides by $\sqrt{7}$, the token width; scaling $W_Q$ by $\sqrt{7}/2$ turns that back into the $\sqrt{4}$ of step 3.)

In [19]:
check = LabelledRead(WIDTH, heads=1, layers=1, n_slots=n_labels, max_sources=256, read="all").eval()
with torch.no_grad():
    check.label.weight.copy_(label_vectors)
    att = check.reads[0].attention
    for lin in (att.q, att.k, att.v, att.o):
        lin.weight.copy_(torch.eye(WIDTH)); lin.bias.zero_()
    att.q.weight.mul_(WIDTH ** 0.5 / 2)
    check.reads[0].norm_readers = check.reads[0].norm_sources = torch.nn.Identity()
    check.mlps[0][-1].weight.zero_(); check.mlps[0][-1].bias.zero_()
    out = check(tokens[None, :, None], labels[None])[0, :, 0]       # (batch 1, photos 7, patches 1, width)
print("test photo:", out[6])
print("matches step 10:", torch.allclose(out, h, atol=1e-5))

test photo: tensor([0.00, 0.00, 2.00, 1.00, 0.22, 0.60, 0.18])
matches step 10: True


## Appendix: learning *what* to compare

So far the similarity was a plain dot product of the raw vectors. That only works if the vectors are already arranged so that "similar" means "same class". Real photo tokens also carry things that have nothing to do with the label: lighting, background, framing.

The fix is the one every attention layer uses. Each photo gets a query and a key through two learned matrices, and the score compares those:

$$q = x_{\text{test}} W_Q, \qquad k_j = x_j W_K, \qquad s_j = q \cdot k_j$$

The values stay the label embeddings. The matrices are not trained on any one task. They are trained across **many** small tasks, each with its own classes and its own random label slots, so the only thing they can learn is *which parts of a photo to compare*. At test time they are fixed, and a new task's answer comes from its context labels.

The toy below trains $W_Q$ and $W_K$ with the read itself as the prediction, to show the idea in isolation. In the model there is no loss on the read: it is trained end to end, through the row stage, the in-context stage and the output head, together with everything else on the image side.

Photos here get 4 more numbers for the **background**: grass, sky, snow, indoors. Backgrounds are loud (3, bigger than any animal feature) and say nothing about the label. Take the animals of step 1 photographed where you would expect them, and a test bird that happens to stand on grass. A plain dot product sees "grass" and votes zebra.

In [20]:
#                        stripes spots wings legs | grass sky snow indoors
farm = torch.tensor([[[2.,     0.,   0.,   2.,     3.,   0.,  0.,  0.],     # zebra on grass    label 0
                      [1.,     0.,   0.,   2.,     3.,   0.,  0.,  0.],     # zebra on grass    label 0
                      [0.,     0.,   2.,   0.,     0.,   3.,  0.,  0.],     # bird in the sky   label 1
                      [0.,     1.,   2.,   0.,     0.,   3.,  0.,  0.],     # bird in the sky   label 1
                      [0.,     2.,   0.,   2.,     0.,   0.,  3.,  0.],     # leopard in snow   label 2
                      [0.,     1.,   0.,   1.,     0.,   0.,  3.,  0.]]])   # leopard in snow   label 2
farm_labels = torch.tensor([[0, 0, 1, 1, 2, 2]])
bird_on_grass = torch.tensor([[[0., 0., 2., 1., 3., 0., 0., 0.]]])
print("scores, plain dot product:", (farm @ bird_on_grass.transpose(1, 2))[0, :, 0].tolist())

scores, plain dot product: [11.0, 11.0, 4.0, 4.0, 2.0, 1.0]


So the model has to learn *what* to compare, and it cannot learn it from this one task. It learns it from many: below, every training task gets fresh random classes (a random prototype in the 4 animal dims each) and fresh random backgrounds (noise of size 2 in the other 4), and its classes land on random label slots. These random tasks are the training data, like the synthetic tables of the real model; the hand-made farm is only for looking at.

In [21]:
WIDTH, N_CLASSES, PER_CLASS, N_TEST = 8, 3, 3, 6    # 8 appearance dims: 4 animal, 4 background

def make_tasks(batch):
    """Tasks of N_CLASSES classes. Each class has a random prototype in the first 4 dims; every photo adds
    nuisance noise (std 2) in the last 4. Labels are random slots per task."""
    protos = torch.randn(batch, N_CLASSES, 4)
    y = torch.arange(N_CLASSES).repeat_interleave(PER_CLASS).expand(batch, -1)          # context classes
    y_test = torch.randint(N_CLASSES, (batch, N_TEST))
    def photos(cls):
        content = protos.gather(1, cls[..., None].expand(-1, -1, 4)) + 0.3 * torch.randn(*cls.shape, 4)
        return torch.cat([content, 2.0 * torch.randn(*cls.shape, 4)], -1)
    slots = torch.stack([torch.randperm(N_CLASSES) for _ in range(batch)])               # class -> label slot
    return photos(y), slots.gather(1, y), photos(y_test), slots.gather(1, y_test)

def labelled_read(test, context, labels, Wq, Wk):
    """test (batch, m, width), context (batch, n, width), labels (batch, n) -> label distribution (batch, m, N_CLASSES)"""
    q, k = test @ Wq, context @ Wk
    w = torch.softmax(q @ k.transpose(1, 2) / WIDTH ** 0.5, dim=-1)
    return w @ F.one_hot(labels, N_CLASSES).float()

def accuracy(Wq, Wk, batch=2000):
    with torch.no_grad():
        xc, yc, xt, yt = make_tasks(batch)
        return (labelled_read(xt, xc, yc, Wq, Wk).argmax(-1) == yt).float().mean().item()

I = torch.eye(WIDTH)
print(f"plain dot product (Wq = Wk = I): accuracy {accuracy(I, I):.2f} on random tasks  (chance is {1 / N_CLASSES:.2f})")
print("the farm, bird on grass, read:", labelled_read(bird_on_grass, farm, farm_labels, I, I)[0, 0])

plain dot product (Wq = Wk = I): accuracy 0.56 on random tasks  (chance is 0.33)
the farm, bird on grass, read: tensor([0.89, 0.08, 0.03])


Now train $W_Q$ and $W_K$. The loss is the cross-entropy of the read: the log of the weight the read puts on the test photo's true label slot. Every step is a fresh batch of tasks.

In [22]:
Wq = (torch.eye(WIDTH) + 0.1 * torch.randn(WIDTH, WIDTH)).requires_grad_()
Wk = (torch.eye(WIDTH) + 0.1 * torch.randn(WIDTH, WIDTH)).requires_grad_()
opt = torch.optim.Adam([Wq, Wk], lr=0.02)
for step in range(1501):
    xc, yc, xt, yt = make_tasks(128)
    r = labelled_read(xt, xc, yc, Wq, Wk)
    loss = F.nll_loss(torch.log(r + 1e-9).flatten(0, 1), yt.flatten())
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 300 == 0:
        print(f"step {step:4d}  loss {loss.item():.3f}  accuracy on new tasks {accuracy(Wq.detach(), Wk.detach()):.2f}")

step    0  loss 1.589  accuracy on new tasks 0.52


step  300  loss 0.347  accuracy on new tasks 0.87


step  600  loss 0.353  accuracy on new tasks 0.87


step  900  loss 0.336  accuracy on new tasks 0.87


step 1200  loss 0.303  accuracy on new tasks 0.86


step 1500  loss 0.298  accuracy on new tasks 0.86


What did it learn? The score is $x_{\text{test}} W_Q W_K^\top x_j^\top$, so $W_Q W_K^\top$ says which dimensions of one photo get compared with which of the other. The top-left block (animal dims) should be large, the bottom-right (background) near zero: the model has learned to ignore the background and compare the animals, without ever being told which dims were which.

In [23]:
M = (Wq @ Wk.T).detach()
print("W_Q W_K^T  (rows: test photo dims, columns: context photo dims; 0-3 animal, 4-7 background)")
print(M)
print(f"mean |entry|, animal block    : {M[:4, :4].abs().mean():.2f}")
print(f"mean |entry|, background block: {M[4:, 4:].abs().mean():.2f}")

W_Q W_K^T  (rows: test photo dims, columns: context photo dims; 0-3 animal, 4-7 background)
tensor([[ 3.78,  0.15, -0.13,  0.05, -0.07, -0.05,  0.01, -0.04],
        [ 0.00,  4.55, -0.16,  0.11,  0.28,  0.05,  0.06,  0.16],
        [ 0.12, -0.07,  4.42, -0.05, -0.04, -0.01, -0.11, -0.18],
        [ 0.02,  0.05,  0.05,  4.26, -0.12,  0.11,  0.02,  0.11],
        [ 0.03, -0.09, -0.04, -0.03, -0.00, -0.02, -0.01,  0.02],
        [ 0.03,  0.05, -0.06, -0.03,  0.00,  0.00, -0.00, -0.00],
        [ 0.00,  0.03,  0.10,  0.00, -0.00,  0.01,  0.02, -0.01],
        [-0.07,  0.04,  0.07, -0.00, -0.00,  0.00,  0.01,  0.01]])
mean |entry|, animal block    : 1.12
mean |entry|, background block: 0.01


Back to the farm, which the model never trained on. The bird on grass is now read as a bird. And it is still in-context: swap the bird and zebra labels and the answer follows, with $W_Q$ and $W_K$ frozen.

In [24]:
with torch.no_grad():
    r = labelled_read(bird_on_grass, farm, farm_labels, Wq, Wk)[0, 0]
    swapped = labelled_read(bird_on_grass, farm, torch.tensor([1, 0, 2])[farm_labels], Wq, Wk)[0, 0]
print("bird on grass, trained read          :", r, "-> label", r.argmax().item())
print("same, zebra and bird labels swapped  :", swapped, "-> label", swapped.argmax().item())

bird on grass, trained read          : tensor([0.03, 0.95, 0.01]) -> label 1
same, zebra and bird labels swapped  : tensor([0.95, 0.03, 0.01]) -> label 0
